In [45]:
from langgraph.graph import StateGraph, START,END
from langchain.tools import tool
from typing import TypedDict, Literal
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
import os

In [46]:
load_dotenv()
ollama_api_key = os.getenv("OLLAMA_API_KEY")

In [47]:
llm=ChatOllama(
    model="gemma4",
    base_url="https://ollama.com",
    client_kwargs={
        "headers": {"Authorization": f"Bearer {ollama_api_key}"}
    }
)

In [48]:
# --------------------------------------------------
# 1. Define State
# --------------------------------------------------
class QAState(TypedDict):
    question: str
    answer: str
    category: str

In [49]:
#define constants
MATH="math"
CALCULATOR="calculator"
GENERAL="general"
CLASSIFY="classify_question"

In [50]:
@tool
def calculator(fist_num:float,second_num:float,operation:str)->float|str:
    """Perform a basic arithmetic operation on two numbers.
    Supported operations: add, sub, mul, div
    """
    if operation=="+":
        return fist_num+second_num
    elif operation=="-":
        return fist_num-second_num
    elif operation=="*":
       return  fist_num*second_num
    elif operation=="/":
        if second_num==0:
            return  "Division by zero is not allowed"
        return  fist_num/second_num
    else: 
        return "Invalid operation"

In [51]:
# --------------------------------------------------
# 2. Classifier Node
# --------------------------------------------------
def classify_question(state:QAState)->QAState:
    
    question=state['question'].lower()
    
    if any(word in question for word in["calculate","plus","minus","multiply","divide","addition","multiplication","division", "+",
        "-",
        "*",
        "/"
]):
        state['category']=MATH #category="math"
    else:
        state['category']=GENERAL #category="general"
    return state

In [52]:
# --------------------------------------------------
# 3. Conditional Routing Function
# --------------------------------------------------
def route_question(state:QAState)->Literal["calculator","general"]:
    if state['category']=="math":
        return CALCULATOR
    return GENERAL
      

In [53]:
# --------------------------------------------------
# 4. Calculator Node
# --------------------------------------------------
def calculator_node(state: QAState)->QAState:
    question=state['question']
    response = llm.invoke(f"Calculator should process: {question}")
    state['answer']=response.content
    return state

In [54]:
# --------------------------------------------------
# 5. General Question Node
# --------------------------------------------------
def general_node(state: QAState):

    question = state["question"]

    response = llm.invoke(f"Answer the following Question: {question}")
    state["answer"] = response.content
    return state

In [55]:
# --------------------------------------------------
# 6. Create Graph
# --------------------------------------------------
graph=StateGraph(QAState)
# add nodes to graph
graph.add_node(CLASSIFY, classify_question)
graph.add_node(CALCULATOR, calculator_node)
graph.add_node(GENERAL, general_node)

# --------------------------------------------------
# 7. START → classify
# --------------------------------------------------
graph.add_edge(START,CLASSIFY)
# --------------------------------------------------
# 8. Conditional Edge
# --------------------------------------------------
graph.add_conditional_edges(CLASSIFY,route_question,{CALCULATOR:CALCULATOR,GENERAL:GENERAL})
# --------------------------------------------------
# 9. Both nodes → END
# --------------------------------------------------
graph.add_edge(CALCULATOR, END)
graph.add_edge(GENERAL, END)

In [56]:
# --------------------------------------------------
# 10. Compile
# --------------------------------------------------
workflow = graph.compile()

In [57]:
initial_state={"question":"What is the capital of France?"}
final_state=workflow.invoke(initial_state)

In [58]:
print(final_state['question'])
print(final_state['category'])
print(final_state['answer'])


What is the capital of France?
general
The capital of France is Paris.


In [59]:
initial_state={"question":"What is the addition of 2 and 3 ?"}
final_state=workflow.invoke(initial_state)

In [60]:
print(final_state['question'])
print(final_state['category'])
print(final_state['answer'])

What is the addition of 2 and 3 ?
math
2 + 3 = 5
